# Tarea 1: Modelación de confiabilidad de puntos de acceso wifi

Este notebook es una **guía de trabajo** para la corroboración computacional solicitada en el enunciado. Debe entregarse completado junto con el informe.

## Integrantes

- TODO: Ingrese el nombre completo y rol USM del integrante 1.
- TODO: Si corresponde, ingrese el nombre completo y rol USM del integrante 2.

## Instrucciones de uso

1. Ejecute las celdas en orden.
2. Complete o modifique las celdas marcadas con TODO.
3. Agregue comentarios breves cuando una decisión requiera justificación.
4. Revise que los resultados del notebook coincidan con el desarrollo matemático del informe.
5. Guarde el notebook con todas las celdas ejecutadas antes de entregarlo.

6. Las interpretaciones y respuestas que no sean código deben escribirse en una celda Markdown titulada como **Celda de respuesta**.

No es necesario convertir este notebook en un programa general ni crear funciones adicionales salvo que ayuden a hacer más claro el análisis.

## 1. Datos y parámetros del problema

Se observan cinco AP hasta su falla permanente. Los tiempos están expresados en días. La misión académica dura 120 días y la confiabilidad mínima aceptable es 0.80.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom

# El formato de los gráficos no afecta los cálculos.

np.set_printoptions(precision=4, suppress=True)


In [ ]:
aps = np.array(["AP01", "AP02", "AP03", "AP04", "AP05"])
causas = np.array([
    "Circuito de alimentación",
    "Módulo de radio",
    "Placa principal",
    "Módulo de radio",
    "Placa principal",
])
tiempos = np.array([110, 150, 205, 250, 295], dtype=float)
mision = 120.0
R_min = 0.80

print("AP:", aps)
print("Causas:", causas)
print("Tiempos [días]:", tiempos)

## 2. Bernoulli y Binomial

Analice el cumplimiento de la misión de $120$ días mediante una variable Bernoulli y estime la probabilidad correspondiente.

Analice el número de AP que continúan funcionando mediante una distribución Binomial. Explique qué información del tiempo de vida se pierde al convertir cada observación en un resultado de misión cumplida o no cumplida.

In [ ]:
# TODO: construya un arreglo/lista de elementos booleanos que indica si cada AP sobrevive o no el tiempo de misión.
sobrevive = None 
# TODO: calcule la probabilidad empírica de completar la misión a partir de sobrevive.
p_hat = None
# TODO: calcule la probabilidad de falla como 1 - p_hat.
p_falla_hat = None
# TODO: calcule la probabilidad P(Y >= 3). Puede usar con scipy.stats.binom 
p_y_al_menos_3 = None

if any(valor is None for valor in [sobrevive, p_hat, p_falla_hat, p_y_al_menos_3]):
    raise NotImplementedError("Complete las cuatro asignaciones de esta celda.")

print("p_hat (completar 120 días):", p_hat)
print("1 - p_hat (fallar durante la misión):", p_falla_hat)
print("P(Y >= 3):", p_y_al_menos_3)

### Celda de respuesta: Bernoulli y Binomial



## 3. Caracterización de los tiempos de falla

Calcule la estimación $\widehat{\mathrm{MTTF}}$, la desviación estándar estimada $\widehat{\sigma}_T$ y el coeficiente de variación $\widehat{C}_V$. Use ddof=1 para la desviación estándar muestral. En esta tarea, $\widehat{\mathrm{MTTF}}$ también se escribe como $\bar t$ y se usará como medida de vida media en las secciones siguientes. Luego interprete qué sugiere la variabilidad observada para la selección del modelo.

In [ ]:
mttf_hat = tiempos.mean()
sigma_hat = tiempos.std(ddof=1)  # TODO: verifique el uso de la desviación muestral
cv_hat = sigma_hat / mttf_hat

print("mttf_hat / t barra [días]:", mttf_hat)
print("sigma_hat [días]:", sigma_hat)
print("C_hat_V:", cv_hat)

### Celda de respuesta: caracterización de los tiempos



## 4. Selección del modelo y sus parámetros

Seleccione entre Exponencial y Weibull en el informe. Considere el valor de $\widehat{C}_V$, el comportamiento esperado de la tasa de riesgo $h(t)$ y las causas de falla.

Si usa Weibull, tome los parámetros indicados en el enunciado. Para $\widehat{C}_V<1$ use $\beta=3.0$ y $\eta=226$ días. Para $\widehat{C}_V>1$ use $\beta=0.75$ y $\eta=170$ días. No debe resolver una ecuación numérica para obtenerlos.

Para Exponencial, calcule $\widehat{\lambda}=1/\widehat{\mathrm{MTTF}}$ y use $\beta=1$ y $\eta=1/\widehat{\lambda}$. Si $\widehat{C}_V$ es igual o muy cercano a 1, el caso Exponencial es el más adecuado.

In [ ]:
modelo = "..."  # TODO: Reemplace "..." por "Weibull" o "Exponencial", según su justificación
lambda_hat = np.nan

if modelo.lower().startswith("weib"):
    if np.isclose(cv_hat, 1.0):
        beta = 1.0
        eta = mttf_hat
        nota_parametros = "C_hat_V cercano a 1: Weibull coincide con Exponencial"
    elif cv_hat < 1:
        beta = 3.0
        eta = 226.0
        nota_parametros = "Valores dados para C_hat_V < 1"
    else:
        beta = 0.75
        eta = 170.0
        nota_parametros = "Valores dados para C_hat_V > 1"
    nombre_modelo = "Weibull"
elif modelo.lower().startswith("exp"):
    beta = 1.0
    lambda_hat = 1 / mttf_hat
    eta = 1 / lambda_hat
    nombre_modelo = "Exponencial"
    nota_parametros = "eta corresponde a la media de los datos"
else:
    raise ValueError("modelo debe ser 'Weibull' o 'Exponencial'")

print("Modelo:", nombre_modelo)
print("beta (adimensional):", beta)
print("eta [días]:", eta)
print("lambda_hat [1/día]:", lambda_hat)
print("Origen de los parámetros:", nota_parametros)

### Celda de respuesta: selección y parámetros



## 5. Funciones de confiabilidad

Complete manualmente las funciones de distribución acumulada $F(t)$, confiabilidad $R(t)$ y tasa de riesgo $h(t)$. Las expresiones deben corresponder al modelo seleccionado en la celda anterior.

In [ ]:
def F(t):
    """Probabilidad de falla acumulada hasta t días."""
    # TODO: implemente F(t) para el modelo seleccionado.
    t = np.asarray(t, dtype=float)
    return None

def R(t):
    """Probabilidad de sobrevivir más allá de t días."""
    # TODO: implemente R(t) usando la relación entre R(t) y F(t).
    return None

def h(t):
    """Tasa de riesgo instantánea en 1/día."""
    # TODO: implemente h(t) con los parámetros beta y eta.
    t = np.asarray(t, dtype=float)
    return None

tiempo_ejemplo = np.array([120.0, 200.0, 295.0])
print("t [días] | F(t) | R(t) | h(t) [1/día]")
for t, f_val, r_val, h_val in zip(tiempo_ejemplo, F(tiempo_ejemplo), R(tiempo_ejemplo), h(tiempo_ejemplo)):
    print(f"{t:7.1f} | {f_val:.4f} | {r_val:.4f} | {h_val:.6f}")

### Celda de respuesta: funciones de confiabilidad



## 6. Misión y reemplazo preventivo

Use $\widehat{\mathrm{MTTF}}$, calculado en la sección de datos, como medida de vida media. El instante de reemplazo preventivo se obtiene de $R(t_p)=R_{\min}$.

In [ ]:
R_120 = float(R(mision))
riesgo_120 = float(h(mision))
riesgo_200 = float(h(200.0))
riesgo_295 = float(h(295.0))
if modelo.lower().startswith("weib"):
    # TODO: calcule tp para el modelo Weibull.
    tp = None
elif modelo.lower().startswith("exp"):
    # TODO: calcule tp para el modelo Exponencial usando lambda_hat.
    tp = None
else:
    raise ValueError("modelo debe ser 'Weibull' o 'Exponencial'")

print("R(120):", R_120)
print("R(tp):", float(R(tp)))
print("tp [días]:", tp)
print("h(120) [1/día]:", riesgo_120)
print("h(200) [1/día]:", riesgo_200)
print("h(295) [1/día]:", riesgo_295)


### Celda de respuesta: misión y reemplazo preventivo



## 7. Gráficos del modelo

El siguiente bloque genera las tres curvas solicitadas: $F(t)$, $R(t)$ y $h(t)$. En el gráfico de confiabilidad se marcan la misión, el umbral mínimo y el reemplazo preventivo del AP individual. Revise etiquetas, unidades, leyenda y escala antes de usar las figuras en el informe.

In [ ]:
t_max = 1.15 * max(tiempos.max(), mision, tp)
t_grid = np.linspace(0, t_max, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(t_grid, F(t_grid), color="tab:red")
axes[0].set_title("Distribución acumulada de falla")
axes[0].set_xlabel("Tiempo [días]")
axes[0].set_ylabel("F(t)")

axes[1].plot(t_grid, R(t_grid), color="tab:blue", label="R(t)")
axes[1].axvline(mision, color="tab:orange", linestyle="--", label="Misión")
axes[1].axvline(tp, color="tab:green", linestyle=":", label="t_p")
axes[1].axhline(R_min, color="black", linestyle="--", label="R_min")
axes[1].scatter([mision, tp], [R_120, R_min], color=["tab:orange", "tab:green"], zorder=3)
axes[1].set_title("Confiabilidad")
axes[1].set_xlabel("Tiempo [días]")
axes[1].set_ylabel("R(t)")
axes[1].legend()

axes[2].plot(t_grid, h(t_grid), color="tab:purple")
axes[2].set_title("Tasa de riesgo")
axes[2].set_xlabel("Tiempo [días]")
axes[2].set_ylabel("h(t) [1/día]")

fig.suptitle(f"Modelo seleccionado: {nombre_modelo}", y=1.03)
fig.tight_layout()
plt.show()

### Celda de respuesta: interpretación de los gráficos



## 8. Exploración

En esta exploración se compara la confiabilidad ajustada $R(t)$ con una supervivencia empírica $\widehat{R}_{\mathrm{emp}}(t)$ construida desde los cinco tiempos observados. Interprete brevemente los resultados obtenidos en esta comparación.

In [ ]:
R_empirica = np.array([np.mean(tiempos > t) for t in t_grid])

plt.figure(figsize=(7, 4))
plt.step(t_grid, R_empirica, where="post", label="Supervivencia empírica")
plt.plot(t_grid, R(t_grid), label=f"Modelo {nombre_modelo}")
plt.scatter(tiempos, np.array([np.mean(tiempos > t) for t in tiempos]), color="black", zorder=3, label="Tiempos observados")
plt.axvline(mision, color="tab:orange", linestyle="--", label="Misión")
plt.xlabel("Tiempo [días]")
plt.ylabel("Confiabilidad / supervivencia")
plt.title("Comparación cualitativa con los datos observados")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

### Celda de respuesta: interpretación de la exploración



## 9. Lista de revisión antes de entregar

- [ ] Se completaron los bloques marcados con TODO.
- [ ] El modelo y sus parámetros coinciden con el informe.
- [ ] Se verificaron $R(120)$, $R(t_p)$, $h(120)$, $h(200)$ y $h(295)$.
- [ ] Los gráficos tienen títulos, ejes, unidades y leyendas legibles.
- [ ] Se interpretaron los resultados de la exploración.
- [ ] El informe contiene las interpretaciones y justificaciones. El notebook no las reemplaza.
- [ ] Se guardó el notebook con las salidas visibles y se entrega junto con el enunciado.